# FedUCA Phase 2: FedAvg Baseline

**Add data:** `ham10000-vit-embeddings-orig` | **GPU T4** | Internet OFF

In [ ]:
import os, json, time, copy, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_phase2"
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
NUM_CLASSES = 7

# Auto-find embeddings regardless of Kaggle mount path
hits = glob.glob("/kaggle/input/**/embeddings_train.npz", recursive=True)
if not hits:
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, files[:3])
    raise FileNotFoundError("embeddings_train.npz not found. Add the dataset first.")
EMBED_DIR = os.path.dirname(hits[0])
print(f"Device    : {DEVICE}")
print(f"Embeddings: {EMBED_DIR}")

In [ ]:
# Load embeddings
def load_split(name):
    d = np.load(f"{EMBED_DIR}/embeddings_{name}.npz", allow_pickle=True)
    return (torch.tensor(d['embeddings'], dtype=torch.float32),
            torch.tensor(d['labels'],     dtype=torch.long))

emb_train, lbl_train = load_split("train")
emb_val,   lbl_val   = load_split("val")
emb_test,  lbl_test  = load_split("test")

print(f"Train: {emb_train.shape} | Val: {emb_val.shape} | Test: {emb_test.shape}")

counts  = torch.bincount(lbl_train, minlength=NUM_CLASSES).float()
weights = 1.0 / counts
weights = (weights / weights.mean()).to(DEVICE)
print(f"Class weights: {weights.cpu().numpy().round(3)}")

In [ ]:
# Head architectures
class LinearHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(768, 7)
    def forward(self, x): return self.fc(x)

class PlainMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 1536), nn.GELU(), nn.LayerNorm(1536), nn.Dropout(0.2),
            nn.Linear(1536, 7))
    def forward(self, x): return self.net(x)

class BottleneckMLP(nn.Module):
    def __init__(self, ratio=4):
        super().__init__()
        b = 768 // ratio
        self.net = nn.Sequential(
            nn.Linear(768, 4*b), nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, b),   nn.GELU(), nn.LayerNorm(b),   nn.Dropout(0.2),
            nn.Linear(b, 4*b),   nn.GELU(), nn.LayerNorm(4*b), nn.Dropout(0.2),
            nn.Linear(4*b, 7))
    def forward(self, x): return self.net(x)

def make_head(ht, ratio=4):
    if ht == "linear":     return LinearHead()
    if ht == "mlp":        return PlainMLP()
    return BottleneckMLP(ratio)

for ht, ratio in [("linear",None),("mlp",None),("bottleneck",2),
                  ("bottleneck",4),("bottleneck",8)]:
    h   = make_head(ht, ratio) if ratio else make_head(ht)
    lbl = f"{ht} N/{ratio}" if ratio else ht
    print(f"  {lbl:20s}: {sum(p.numel() for p in h.parameters()):>12,} params")

In [ ]:
# Utilities: partition, evaluate, local_train, fedavg
def dirichlet_partition(labels, K, alpha, seed):
    rng = np.random.default_rng(seed)
    idx = [[] for _ in range(K)]
    for c in range(int(labels.max()) + 1):
        ci = np.where(labels.numpy() == c)[0]
        rng.shuffle(ci)
        props  = rng.dirichlet(np.full(K, alpha))
        splits = (np.cumsum(props) * len(ci)).astype(int)[:-1]
        for k, chunk in enumerate(np.split(ci, splits)):
            idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

@torch.no_grad()
def evaluate(head, embs, labels):
    head.eval()
    p, t = [], []
    for x, y in DataLoader(TensorDataset(embs, labels), batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu())
        t.append(y)
    p = torch.cat(p).numpy()
    t = torch.cat(t).numpy()
    f1s = []
    for c in range(NUM_CLASSES):
        tp = ((p==c)&(t==c)).sum(); fp = ((p==c)&(t!=c)).sum(); fn = ((p!=c)&(t==c)).sum()
        pr = tp/(tp+fp+1e-9);       rc = tp/(tp+fn+1e-9)
        f1s.append(2*pr*rc/(pr+rc+1e-9))
    return {"accuracy": float((p==t).mean()), "macro_f1": float(np.mean(f1s)),
            "f1_per_class": [float(x) for x in f1s]}

def local_train(head, embs, labels, lr=5e-3, epochs=5, bs=32):
    h    = copy.deepcopy(head).to(DEVICE); h.train()
    opt  = optim.Adam(h.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        for x, y in DataLoader(TensorDataset(embs.to(DEVICE), labels.to(DEVICE)),
                               batch_size=bs, shuffle=True):
            opt.zero_grad(); crit(h(x), y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def fedavg(states, sizes):
    total = sum(sizes)
    return {k: sum((sizes[i]/total)*states[i][k].float()
                   for i in range(len(sizes)))
            for k in states[0]}

print("Utilities ready.")

In [ ]:
# Federated training loop
def run_federated(head_type, ratio, alpha, seed,
                  n_rounds=60, K=4, epochs=5, lr=5e-3, bs=32):
    partition = dirichlet_partition(lbl_train, K, alpha, seed)
    c_embs    = [emb_train[idx] for idx in partition]
    c_labels  = [lbl_train[idx] for idx in partition]
    sizes     = [len(idx)       for idx in partition]
    head      = make_head(head_type, ratio).to(DEVICE)

    history = []; best_val = -1; best_round = 0; best_state = None
    t0 = time.time()
    for rnd in range(1, n_rounds + 1):
        states = [local_train(head, c_embs[k], c_labels[k], lr, epochs, bs)
                  for k in range(K)]
        head.load_state_dict(fedavg(states, sizes))
        vm = evaluate(head, emb_val.to(DEVICE), lbl_val)
        history.append(vm['macro_f1'])
        if vm['macro_f1'] > best_val:
            best_val = vm['macro_f1']; best_round = rnd
            best_state = copy.deepcopy(head.state_dict())
        if rnd % 10 == 0 or rnd == 1:
            print(f"  r{rnd:3d} val_F1={vm['macro_f1']:.4f} "
                  f"best={best_val:.4f}@r{best_round} | {time.time()-t0:.0f}s")

    head.load_state_dict(best_state)
    tm = evaluate(head, emb_test.to(DEVICE), lbl_test)

    # Save weights for Phase 5
    tag = f"{head_type}_r{ratio}" if ratio else head_type
    torch.save(best_state,
               os.path.join(RESULTS_DIR, f"weights_{tag}_a{alpha}_s{seed}.pt"))
    return {"head_type": head_type, "ratio": ratio, "alpha": alpha, "seed": seed,
            "best_round": best_round, "best_val_f1": best_val,
            "val_history": history, "test": tm}

print("run_federated() ready.")

In [ ]:
# Main ablation
print("="*55)
print("PHASE 2: FedAvg ablation (4 heads x 3 alpha x 5 seeds)")
print("="*55)

HEADS  = [("linear",None), ("mlp",None), ("bottleneck",4), ("bottleneck",8)]
ALPHAS = [1000, 1, 0.1]
SEEDS  = [0, 1, 2, 3, 4]
all_results = []

for alpha in ALPHAS:
    print(f"\n=== alpha={alpha} ===")
    for ht, ratio in HEADS:
        tag = f"{ht}_r{ratio}" if ratio else ht
        seed_res = []
        for seed in SEEDS:
            print(f"  [{tag}] alpha={alpha} seed={seed}")
            res = run_federated(ht, ratio, alpha, seed)
            seed_res.append(res)
            all_results.append(res)
            with open(os.path.join(RESULTS_DIR,
                                   f"fedavg_{tag}_a{alpha}_s{seed}.json"), "w") as f:
                json.dump(res, f, indent=2)
        f1s = [r['test']['macro_f1'] for r in seed_res]
        print(f"  >>> {tag} alpha={alpha}: F1={np.mean(f1s):.3f} +/- {np.std(f1s):.3f}")

with open(os.path.join(RESULTS_DIR, "all_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
print("\nAll results saved.")

In [ ]:
# Summary table
print("\n" + "="*55)
print("SUMMARY (alpha=1000) — compare to paper Table 5")
print("="*55)
print(f"{'Head':<20} {'F1 mean':>9} {'F1 std':>8} {'Acc mean':>10}")
print("-"*47)

paper = {"linear":0.679, "mlp":0.728, "bottleneck_4":0.719, "bottleneck_8":0.710}
for ht, ratio in [("linear",None),("mlp",None),("bottleneck",4),("bottleneck",8)]:
    rows = [r for r in all_results
            if r['alpha']==1000 and r['head_type']==ht and r['ratio']==ratio]
    f1s  = [r['test']['macro_f1'] for r in rows]
    accs = [r['test']['accuracy']  for r in rows]
    key  = f"{ht}_{ratio}" if ratio else ht
    disp = f"Bottleneck N/{ratio}" if ht=="bottleneck" else ht.capitalize()
    print(f"{disp:<20} {np.mean(f1s):>9.3f} {np.std(f1s):>8.3f} "
          f"{np.mean(accs):>10.3f}  (paper ref: {paper.get(key,0):.3f})")

print()
print("Results saved to:", RESULTS_DIR)
print("NEXT: Save output as dataset 'feduca-phase2-results', then run Phase 3")